#  Preprocessing and Feature Engineering

* In the case of most classification and regression algorithms, you want to get your data into a column of type Double to represent the label and a column of type Vector (either dense or sparse) to represent the features.
* In the case of recommendation, you want to get your data into a column of users, a column of items (say movies or books), and a column of ratings.
* In the case of unsupervised learning, a column of type Vector (either dense or sparse) is needed to represent the features.
* In the case of graph analytics, you will want a DataFrame of vertices and a DataFrame of edges.

In [ ]:
import os
import s3fs

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = (SparkSession.builder
         .appName('my_spark_app')
         .getOrCreate())

In [ ]:
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

In [ ]:
fs.ls('fgao-ensae/Data_spark/ML_Lib')

In [ ]:
BUCKET = 'fgao-ensae'
FILE_KEY_S3 = "Data_spark/ML_Lib/simple-ml"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"

In [ ]:
simpleDF = spark.read.json(s3_path)
simpleDF.printSchema()

# simpleDF = spark.read.json("/databricks-datasets/definitive-guide/data/simple-ml")

## 1. estimator et transformer

1. fit(), combien de passage sur les données ?
2. transform() ==> transformer
3. fit_transform() ==> estimator
4. pd.cut() ?
5. pd.qcut() ?

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer

# Jeu de données avec des valeurs manquantes (NaN)
X_train = np.array([
    [1, 2],
    [np.nan, 3],
    [7, 6]
])

X_test = np.array([
    [np.nan, 2],
    [6, np.nan]
])

# Création d’un imputer qui remplace les NaN par la moyenne de la colonne
imputer = SimpleImputer(strategy='mean')

# 1. fit() : on calcule les moyennes des colonnes à partir des données d'entraînement
imputer.fit(X_train)

# 2. transform() : on applique la transformation (remplacement des NaN)
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 3. fit_transform() : combine fit() et transform() en une seule étape
X_train_imputed_alt = imputer.fit_transform(X_train)

print("X_train après imputation :\n", X_train_imputed)
print("X_test après imputation :\n", X_test_imputed)


In [ ]:
from pyspark.ml.feature import Bucketizer, QuantileDiscretizer, MinMaxScaler, OneHotEncoder

Un par un, estimator(2 passages de données) or transformer (1 passage) ?
* Bucketizer = pd.cut()
* QuantileDiscretizer = pd.qcut()

Ecrire tous les titres d'abord pour donner une vue globale

## 2. Continuous numeric features
### Bucketing and QuantileDiscretizer
QuantileDiscretizer et Bucketizer dans PySpark ne fonctionnent que sur les colonnes de type DoubleType ou FloatType, et non sur les types LongType ou IntegerType.

In [1]:
from pyspark.ml.feature import Bucketizer, QuantileDiscretizer

In [ ]:
buket = Bucketizer(
    inputCol='value2',
    outputCol='class',
    splits=[10, 20, 30, 40]
)

buket.transform(simpleDF).show()

In [ ]:
# elève
qd = QuantileDiscretizer(
    inputCol='value2',
    outputCol='quantile'
)

qd.fit(simpleDF).transform(simpleDF).show()

## 3. Categorical Features
### StringIndexer and OHE

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [ ]:
si = StringIndexer(
    inputCol= 'lab',
    outputCol= 'label'
)
si.fit(simpleDF).transform(simpleDF).show()

### 3.2. One-Hot Encoding

Tu peux poser la question : pourquoi ohe doit se faire en 2 étapes ?

Spark’s restriction of the OneHotEncoder to work only with numeric categories (via StringIndexer first) is primarily due to design considerations for scalability, flexibility, and performance in distributed systems.

StringIndexer converts categorical string values to numeric indices (ordinal labels), which is essential for many machine learning algorithms in Spark, such as decision trees or random forests, which don't require one-hot encoding but can work with indexed labels.


1. Separation of Concerns:

StringIndexer and OneHotEncoder are designed as separate transformations to handle distinct tasks in a modular way.
StringIndexer converts categorical string values to numeric indices (ordinal labels), which is essential for many machine learning algorithms in Spark, such as decision trees or random forests (et aussi Boosting, Bayesian), which don't require one-hot encoding but can work with indexed labels.
Why ? Contrairement à des modèles linéaires (comme la régression logistique), qui supposent une relation linéaire entre les variables, les arbres ne se préoccupent pas des distances numériques entre les indices (par exemple, que 2 soit plus grand que 1). Ils testent des règles comme : "Si la valeur de la caractéristique est égale à 1, alors...". Pour un arbre de décision, il peut directement utiliser ces indices pour créer des règles comme :
"Si Color = 0 (Red), alors..."
"Si Color = 1 (Blue), alors..."



OneHotEncoder takes the numeric indices and produces one-hot encoded vectors, which are necessary for algorithms that expect continuous numerical input.
By splitting these two tasks, Spark allows for more fine-grained control over the preprocessing pipeline. For example, you may want to reuse the indexed labels for different algorithms, or only use one-hot encoding for certain parts of the data. This flexibility is useful in large, complex pipelines.

3. Optimized Distributed Processing:
Spark is designed to process massive datasets across distributed clusters. By restricting OneHotEncoder to work only with numeric indices, Spark reduces the computational complexity and communication overhead between nodes.
String-to-index conversion is a non-trivial operation because it requires handling large, potentially unordered sets of string categories. For large datasets distributed across multiple nodes, handling strings can introduce additional serialization and network communication overhead.
Numeric operations (like creating one-hot encodings from indices) are more efficient and scalable in a distributed environment because they use less memory and are faster to process. Spark can optimize the transformation steps internally when working with numeric data, especially during distributed computation.
4. Scalability for Big Data:
In distributed systems, working with numeric data is faster and easier to parallelize than working with string data. Numeric indices are lightweight and can be stored and processed more efficiently than variable-length strings.
For example, Spark can handle indexing in parallel, converting strings to numbers at the distributed level. Once this conversion is done, the OneHotEncoder step is much faster because it only operates on integers. This separation minimizes overhead in large-scale datasets where strings can be much larger than the corresponding integer indices.

In [ ]:
si = StringIndexer(
    inputCol= 'color',
    outputCol='color_idx'
)

ohe = OneHotEncoder(
    inputCol = 'color_idx',
    outputCol = 'color_ohe'
)

simple_df_inx = si.fit(simpleDF).transform(simpleDF)
ohe.fit(simple_df_inx).transform(simple_df_inx).show()

- So (2,[],[]) means '00' 
- 2,[1],[1.0] with 1.0 at position 1 (01)
- (2,[0],[1.0]) with 1.0 at position 0 (10)


"color_ohe" is represented in **sparse format**. In this format the zeros of a vector are not printed. 
(2,[1],[1.0])

- The first value (2) shows the length of the vector, here there are 3 value so the length of the vector is always 2 : position 0 and position 1
- The third value is an array that tells which numbers (except 0) are found. 
- the second value is an array that lists these numbers (except 0 are found in with position (dans notre exemple, seulement position 0 or 1 existe cause length of the vector is 2) *zero or more indices where non-zero entries are found*.

## 4. First Pipeline

In [ ]:
from pyspark.ml import Pipeline

In [ ]:
pip = Pipeline(stages=[
    StringIndexer(inputCol= 'lab',outputCol= 'label'),
    StringIndexer(inputCol= 'color', outputCol='color_idx'),
    OneHotEncoder(inputCol = 'color_idx', outputCol = 'color_ohe')
]
)

In [ ]:
pip.fit(simpleDF).transform(simpleDF).show(5)

## 5. VectorAssembler 

In [ ]:
from pyspark.ml.feature import VectorAssembler

In [ ]:
va = VectorAssembler(
    inputCols=['value1', 'value2', 'color_ohe',],
    outputCol='features'
)

In [2]:
va.transform(
    pip.fit(simpleDF).transform(simpleDF)
).show()

NameError: name 'va' is not defined

## 6. Generalized Pipeline

In [ ]:
df = simpleDF.select("*")

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

In [ ]:
[field.name for field in df.schema.fields]

In [ ]:
[field.dataType.typeName() for field in df.schema.fields]

In [ ]:
label = 'lab'

numeric_cols = [field.name for field in df.schema.fields
               if field.dataType.typeName() in ['double', 'long', 'float', 'interger'] and field.name != label
               ]

numeric_cols

float
* 32 bits (4 octets)
* Environ 7 chiffres significatifs
* Moins précis, mais plus léger en mémoire

double
* 64 bits (8 octets)
* Environ 15 à 17 chiffres significatifs
* Plus précis, donc préférable pour des calculs sensibles ou des valeurs à grande échelle
* C’est le type par défaut en PySpark pour les décimales

In [ ]:
categorical_cols = [field.name for field in df.schema.fields
               if field.dataType.typeName() in ['string'] and field.name != label
               ]

categorical_cols

In [ ]:
numeric_pipeline = Pipeline(stages = [
    VectorAssembler().setInputCols(numeric_cols).setOutputCol('numeric_cols_vec'),
    StandardScaler().setInputCol('numeric_cols_vec').setOutputCol('numeric_features')
])

categorical_stages = []
for col in categorical_cols :     
    index_col = f"{col}_idx"
    ohe_col = f"{col}_ohe"
    print(col)
    print(index_col)
    print(ohe_col)
    categorical_stages.append(StringIndexer(inputCol=col, outputCol=index_col))
    categorical_stages.append(OneHotEncoder(inputCol=index_col, outputCol=ohe_col))

In [ ]:
categorical_stages

In [ ]:
numeric_pipeline

In [ ]:
categorical_pipeline = Pipeline(stages=categorical_stages)

In [ ]:
categorical_pipeline

In [ ]:
assembler_input_cols = ['numeric_features'] + [f"{col}_ohe" for col in categorical_cols]
assembler_input_cols

In [ ]:
# Main pipeline

final_pipeline = Pipeline(stages=[
    numeric_pipeline,
    categorical_pipeline,
    VectorAssembler(inputCols=assembler_input_cols, outputCol='features'),
    StringIndexer(inputCol=label, outputCol = 'label')
])

In [ ]:
categorical_pipeline.fit(df).transform(df).show()

In [ ]:
final_pipeline.fit(df).transform(df).show()

In [ ]:
final_pipeline.fit(df).transform(df).select('features', 'label').show()

## Exo 

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler


label = 'lab'

# numeric_col = 

# categorical_cols = 

# numeric_pipeline = 
# )
# categorical_pipeline = 

assmbler_input_cols = ['numeric_features'] + [f"{col}_ohe" for col in categorical_cols]
assmbler_input_cols

final_pipeline = Pipeline(stages = [
                         numeric_pipeline,
                         categorical_pipeline,
                         VectorAssembler(inputCols=assmbler_input_cols, outputCol= 'features'),
                         StringIndexer(inputCol= label, outputCol = 'label')
                         ])

final_pipeline.fit(df).transform(df).show()